### EBA detects structural in absence of clear sequence identity

In [ ]:
import pickle
import os
import scipy.stats
import matplotlib.pyplot as plt
from matplotlib import cm
from matplotlib.colors import ListedColormap, LinearSegmentedColormap
viridis = cm.get_cmap('viridis', 256)

In [ ]:
working_dir = '.'
results_path = os.path.join(working_dir, 'results')
data_dir = os.path.join(working_dir,'data')
sequence_file = os.path.join(data_dir, 'sequences.fasta')
tm_scores_path = os.path.join(results_path, 'TM_scores.pickle')

Load PISCES pairs TM-scores 

In [ ]:
with open(tm_scores_path, 'rb') as handle:
    all_tm_scores = pickle.load(handle)
    
print('{} TM scores loaded'.format(len(all_tm_scores)))

Remove reduntant pairs ( TM(a,b)=TM(b,a) )

In [ ]:
tm_scores = dict()
for p in all_tm_scores:
    if (p[1],p[0]) not in tm_scores.keys():
        tm_scores[p]=all_tm_scores[p]
        
print('{} TM scores filtred'.format(len(tm_scores)))

Load sequences

In [ ]:
sequences = dict()
with open(sequence_file, 'r') as file:
    for line in file:
        if line[0]=='>':
            pdb_id = line.split()[0][1:]
            sequences[pdb_id] = ''
        else:
            sequences[pdb_id]+=line.strip()

seq_length = {seq_id:len(sequences[seq_id])for seq_id in sequences}

Load predictions

In [ ]:
results = dict()
for scores in os.listdir(results_path):
    if scores.split('_')[0]=='TM':
        continue
    score_file = os.path.join(results_path, scores)
    with open(score_file, 'rb') as handle:
        print('loading: {}'.format(scores))
        results[scores.split('.')[0]] = pickle.load(handle)

### Define sorting functions

In [ ]:
def sort_EBA_results(model, tm_scores):
    ordered_results = {'min':list(), 'max':list(), 'raw':list()}
    gt_ordered = {'tm_min':list(), 'tm_max':list(), 'r':list()}
    
    for p in results['{}_EBA'.format(model)]:
        short = min(seq_length[p[0]],seq_length[p[1]])
        long = max(seq_length[p[0]],seq_length[p[1]])
        
        #skip short sequences
        if short<75:
            continue
        
        if p in tm_scores.keys():
            gt_ordered['tm_min'].append(min(tm_scores[p]['TM1'], tm_scores[p]['TM2']))
            gt_ordered['tm_max'].append(max(tm_scores[p]['TM1'], tm_scores[p]['TM2']))
            gt_ordered['r'].append(short/long)
            
            ordered_results['min'].append(results['{}_EBA'.format(model)][p]['EBA_min'])
            ordered_results['max'].append(results['{}_EBA'.format(model)][p]['EBA_max'])
            ordered_results['raw'].append(results['{}_EBA'.format(model)][p]['EBA_raw'])
            
        else:
            continue
            
    return ordered_results,gt_ordered

In [ ]:
def sort_AD_results(model, tm_scores):
    ordered_results = list()
    gt_ordered = {'tm_min':list(), 'tm_max':list(), 'r':list()}
    
    for p in results['{}_AD'.format(model)]:
        short = min(seq_length[p[0]],seq_length[p[1]])
        long = max(seq_length[p[0]],seq_length[p[1]])
        
        #skip short sequences
        if short<75:
            continue
        
        if p in tm_scores.keys():
            gt_ordered['tm_min'].append(min(tm_scores[p]['TM1'], tm_scores[p]['TM2']))
            gt_ordered['tm_max'].append(max(tm_scores[p]['TM1'], tm_scores[p]['TM2']))
            gt_ordered['r'].append(short/long)
            
            ordered_results.append(results['{}_AD'.format(model)][p].item())
            
        else:
            continue
            
    return ordered_results,gt_ordered

## Sort results

In [ ]:
ESMb1_EBA_results, ESMb1_EBA_gt = sort_EBA_results('ESMb1', tm_scores)
ProtT5_EBA_results, ProtT5_EBA_gt = sort_EBA_results('ProtT5', tm_scores)

In [ ]:
ESMb1_AD_results, ESMb1_AD_gt = sort_AD_results('ESMb1', tm_scores)
ProtT5_AD_results, ProtT5_AD_gt = sort_AD_results('ProtT5', tm_scores)

### ESM-b1 Spearman correlation

In [ ]:
print('EBA\n')
print('tm_min:', scipy.stats.spearmanr(ESMb1_EBA_results['min'], ESMb1_EBA_gt['tm_min']))
print('tm_max:', scipy.stats.spearmanr(ESMb1_EBA_results['max'], ESMb1_EBA_gt['tm_max']))

In [ ]:
print('AD\n')
print('tm_min:', scipy.stats.spearmanr(ESMb1_AD_results, ESMb1_AD_gt['tm_min']))
print('tm_max:', scipy.stats.spearmanr(ESMb1_AD_results, ESMb1_AD_gt['tm_max']))

### ProtT5 Spearman correlations

In [ ]:
print('EBA\n')
print('tm_min:', scipy.stats.spearmanr(ProtT5_EBA_results['min'], ProtT5_EBA_gt['tm_min']))
print('tm_max:', scipy.stats.spearmanr(ProtT5_EBA_results['max'], ProtT5_EBA_gt['tm_max']))

In [ ]:
print('AD\n')
print('tm_min:', scipy.stats.spearmanr(ProtT5_AD_results, ProtT5_AD_gt['tm_min']))
print('tm_max:', scipy.stats.spearmanr(ProtT5_AD_results, ProtT5_AD_gt['tm_max']))

### Plot example: ProtT5

In [ ]:
colors = ProtT5_EBA_gt['r']
plt.xlabel('TM min scores')
plt.ylabel('EBA min')
plt.scatter(ProtT5_EBA_gt['tm_min'], 
            ProtT5_EBA_results['min'],
            alpha=0.5,
            c=colors,
            cmap= viridis,
            vmin=0, vmax=1)
plt.colorbar()